# X/Twitter Scraper for Google Colab

This notebook provides an easy-to-use interface for scraping X (Twitter) data using the twscrape library.

**⚠️ Important Notes:**
- Twitter scraping requires accounts to be configured
- Some features may be rate-limited
- Respect Twitter's terms of service and robots.txt
- Use responsibly and ethically

---

## 1. Setup and Dependencies

First, let's install the required packages and set up the environment.

In [ ]:
# Install required packages
!pip install twscrape pandas nest-asyncio

# Import libraries
import asyncio
import json
import sys
from pathlib import Path
from typing import List, Dict, Any
from contextlib import aclosing
import nest_asyncio
import pandas as pd

# Enable nested event loops for Colab
nest_asyncio.apply()

from twscrape import API, gather
from twscrape.logger import set_log_level

# Set log level
set_log_level("INFO")

print("✅ Setup complete!")

## 2. Account Setup

Configure your Twitter accounts for scraping.

In [ ]:
#@title Twitter Scraper Class

class TwitterScraper:
    def __init__(self):
        self.api = API()

    async def check_accounts(self):
        """Check available accounts"""
        try:
            accounts = await self.api.pool.get_all()
            if not accounts:
                return False, "No accounts configured"

            active_accounts = [acc for acc in accounts if acc.active]
            return len(active_accounts) > 0, f"{len(active_accounts)} active accounts available"
        except Exception as e:
            return False, f"Error checking accounts: {e}"

    async def search_tweets(self, query: str, limit: int = 100) -> List[Dict[str, Any]]:
        """Search for tweets using a query"""
        tweets = []
        try:
            async with aclosing(self.api.search(query, limit=limit)) as gen:
                async for tweet in gen:
                    tweet_data = {
                        'id': tweet.id,
                        'url': tweet.url,
                        'date': tweet.date.isoformat() if tweet.date else None,
                        'content': tweet.rawContent,
                        'username': tweet.user.username,
                        'display_name': tweet.user.displayname,
                        'verified': getattr(tweet.user, 'verified', False),
                        'followers_count': getattr(tweet.user, 'followersCount', 0),
                        'reply_count': getattr(tweet, 'replyCount', 0),
                        'retweet_count': getattr(tweet, 'retweetCount', 0),
                        'like_count': getattr(tweet, 'likeCount', 0),
                        'hashtags': [getattr(tag, 'text', str(tag)) for tag in getattr(tweet, 'hashtags', []) or []]
                    }
                    tweets.append(tweet_data)
                    if len(tweets) % 10 == 0:
                        print(f"Collected {len(tweets)}/{limit} tweets")
        except Exception as e:
            print(f"Error searching tweets: {e}")

        return tweets

    async def get_user_tweets(self, username: str, limit: int = 50) -> List[Dict[str, Any]]:
        """Get recent tweets from a user"""
        try:
            user = await self.api.user_by_login(username)
            tweets = await gather(self.api.user_tweets(user.id, limit=limit))
            return [{
                'id': tweet.id,
                'url': tweet.url,
                'date': tweet.date.isoformat() if tweet.date else None,
                'content': tweet.rawContent,
                'reply_count': tweet.replyCount,
                'retweet_count': tweet.retweetCount,
                'like_count': tweet.likeCount
            } for tweet in tweets]
        except Exception as e:
            print(f"Error getting tweets for {username}: {e}")
            return []

# Initialize scraper
scraper = TwitterScraper()

# Check account status
accounts_ok, message = await scraper.check_accounts()
print(f"Account Status: {message}")

if not accounts_ok:
    print("\nTo set up accounts:")
    print("1. Create an accounts.txt file with your Twitter credentials")
    print("2. Format: username:password:email:email_password")
    print("3. Upload it to Colab")
else:
    print("Ready to scrape!")

### Upload Accounts File

In [ ]:
# Upload accounts file
from google.colab import files

try:
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        print(f'Uploaded: {filename}')
        if filename == 'accounts.txt':
            !mv {filename} .
            print("Accounts file ready!")
            break
except:
    print("File upload not available or cancelled")

### Add Account Manually

In [ ]:
#@title Add Twitter Account

# Uncomment and fill in your credentials
# username = "your_username"  #@param {type:"string"}
# password = "your_password"  #@param {type:"string"}
# email = "your_email@gmail.com"  #@param {type:"string"}
# email_password = "email_password"  #@param {type:"string"}

# If you filled in the credentials above, uncomment the next lines:
# await scraper.api.pool.add_account(username, password, email, email_password)
# await scraper.api.pool.login_all()
# print("Account added and logged in!")

print("Please uncomment and fill in your credentials above, then run this cell.")

## 3. Data Collection

Choose what type of data you want to collect.

In [ ]:
#@title Search Tweets

search_query = "AI"  #@param {type:"string"}
search_limit = 50  #@param {type:"slider", min:10, max:200, step:10}
run_search = False  #@param {type:"boolean"}

if run_search and search_query.strip():
    print(f"Searching for: '{search_query}' (limit: {search_limit})")
    tweets = await scraper.search_tweets(search_query, search_limit)
    
    if tweets:
        print(f"Collected {len(tweets)} tweets")
        
        # Save to JSON file
        filename = f"tweets_{search_query.replace(' ', '_')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tweets, f, indent=2, ensure_ascii=False)
        print(f"Saved to: {filename}")
        
        # Convert to DataFrame for preview
        df = pd.DataFrame(tweets)
        print("\nPreview:")
        display(df.head())
        
    else:
        print("No tweets found")
else:
    print("Configure search parameters above and set 'run_search' to True")

In [ ]:
#@title Get User Tweets

username = "elonmusk"  #@param {type:"string"}
tweet_limit = 25  #@param {type:"slider", min:5, max:100, step:5}
run_user_scrape = False  #@param {type:"boolean"}

if run_user_scrape and username.strip():
    print(f"Getting tweets for: @{username} (limit: {tweet_limit})")
    tweets = await scraper.get_user_tweets(username, tweet_limit)
    
    if tweets:
        print(f"Collected {len(tweets)} tweets")
        
        # Save to JSON file
        filename = f"tweets_{username}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tweets, f, indent=2, ensure_ascii=False)
        print(f"Saved to: {filename}")
        
        # Convert to DataFrame for preview
        df = pd.DataFrame(tweets)
        print("\nPreview:")
        display(df.head())
        
    else:
        print("No tweets found")
else:
    print("Enter a username above and set 'run_user_scrape' to True")

## 4. Export Data

Download your collected data.

In [ ]:
#@title Download Files

from google.colab import files
import os

# List available JSON files
json_files = [f for f in os.listdir('.') if f.endswith('.json')]

if json_files:
    print("Available files:")
    for file in json_files:
        size = os.path.getsize(file) / 1024
        print(f"  - {file}: {size:.1f} KB")
    
    # Download all JSON files
    for file in json_files:
        files.download(file)
        
    print(f"\nDownloaded {len(json_files)} files")
else:
    print("No JSON files found. Please collect some data first.")